<a href="https://colab.research.google.com/github/kamejoko80/optispeech/blob/henry_rk3588/notebooks/henry_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Python 3.11

In [ ]:
# 1. Install Python 3.11 and Dev tools globally
!apt-get update
!apt-get install python3.11 python3.11-dev python3.11-distutils -y

# 2. Register both versions in the 'update-alternatives' system
# We give Python 3.11 a higher priority (2) than Python 3.12 (1)
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1

# 3. Fix Pip (changing Python versions often breaks the global pip link)
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --force-reinstall

# 4. Verify the change
!python3 --version


# Environment Settings

In [ ]:
%cd /content

!pip install -U pip
!git clone -b henry_rk3588 https://github.com/kamejoko80/optispeech.git

# Install OptiSpeech
%cd /content/optispeech
!pip install -e .

# Install some extra dependencies
!pip install transformers -U
!pip install onnxruntime soundfile numpy -U
!pip install torch torchvision torchaudio -U
!python3 -c "import torch; print(torch.__version__)"

%cd /content/optispeech/rknn_rk3588

# Create models and out folders
import os
folders = ['models', 'out']
for folder in folders:
    if os.path.exists(folder):
        print(f"✅ Folder '{folder}' already exists.")
    else:
        os.makedirs(folder)
        print(f"📁 Folder '{folder}' was created.")

        # Download the prestrained checkpoint
        from huggingface_hub import hf_hub_download
        repo_id = "henrydang80/optispeech"
        filename = "checkpoints/lightspeech/en-us/mike-checkpoint_epoch-729_step-305000.ckpt"
        path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir="models", local_dir_use_symlinks=False)
        print("Saved to:", path)




# Model Inference Example

In [ ]:
%cd /content/optispeech/rknn_rk3588
!python3 test_pytorch.py

from IPython.display import Audio
Audio('output.wav')

# Prepare Dataset For The Model Training

In [ ]:
%cd /content/optispeech

# Create datasets folder
import os
folders = ['datasets']
for folder in folders:
    if os.path.exists(folder):
        print(f"✅ Folder '{folder}' already exists.")
    else:
        os.makedirs(folder)
        print(f"📁 Folder '{folder}' was created.")

        # Dowload the datasets
        %cd datasets
        !wget -O LJSpeech-1.1.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
        !tar -xjf LJSpeech-1.1.tar.bz2

        # Create the lightning-hydra-template folder
        !mkdir -p ljs_split/train/wav ljs_split/val/wav

        # Create the lightning-hydra-template folder
        from pathlib import Path
        import random

        src = Path("LJSpeech-1.1").resolve()
        meta = src / "metadata.csv"
        wavs = src / "wavs"

        out = Path("ljs_split").resolve()
        (out / "train" / "wav").mkdir(parents=True, exist_ok=True)
        (out / "val" / "wav").mkdir(parents=True, exist_ok=True)

        lines = meta.read_text(encoding="utf-8").splitlines()
        random.seed(1234)
        random.shuffle(lines)

        val_n = 500
        val_lines = lines[:val_n]
        train_lines = lines[val_n:]

        def write_split(split_name: str, split_lines: list[str]):
            out_meta = out / split_name / "metadata.csv"
            out_wavdir = out / split_name / "wav"

            rows_2col = []
            missing = 0

            for ln in split_lines:
                parts = ln.split("|")
                if len(parts) < 2:
                    continue

                utt = parts[0].strip()
                text = parts[2].strip() if len(parts) >= 3 else parts[1].strip()
                rows_2col.append(f"{utt}|{text}")

                src_wav = (wavs / f"{utt}.wav").resolve()
                dst_wav = out_wavdir / f"{utt}.wav"

                if not src_wav.exists():
                    missing += 1
                    continue

                if dst_wav.exists() or dst_wav.is_symlink():
                    continue

                dst_wav.symlink_to(src_wav)  # ABSOLUTE target

            out_meta.write_text("\n".join(rows_2col) + "\n", encoding="utf-8")
            return len(rows_2col), missing

      n_train, m_train = write_split("train", train_lines)
      n_val, m_val = write_split("val", val_lines)

      print("Wrote:", out)
      print("train:", n_train, "missing wav:", m_train)
      print("val:  ", n_val, "missing wav:", m_val)
